In [97]:
import warnings
warnings.filterwarnings("ignore")

In [98]:
import pandas as pd
import xml.etree.ElementTree as et

# **Data processing**

**XML Parsing**

In [99]:
sentiment_label_encode = {
    'NONE': 0,
    'P': 1,
    'N': 2,
    'NEU': 3
}

In [100]:
def xml_2_dataframe(path):
    xtree = et.parse(path)
    xroot = xtree.getroot()

    df = pd.DataFrame({
        'id': pd.Series(dtype='str'),
        'label': pd.Series(dtype='str'),
        'text': pd.Series(dtype='str')
    })

    for tweet in xroot:
        tweet_id = tweet.find('tweetid').text
        content = tweet.find('content').text
        sentiment = tweet.find('sentiment').find('polarity').find('value').text
        new_input = {'id':tweet_id, 'label': sentiment, 'text': content}
        df = df.append(new_input, ignore_index=True)

    return df

In [101]:
train_df = xml_2_dataframe('data/TASS2017_T1_training.xml')
dev_df = xml_2_dataframe('data/TASS2017_T1_training.xml')
test_df = xml_2_dataframe('data/TASS2017_T1_test.xml')

In [102]:
print(train_df.head())

                   id label                                               text
0  768213876278165504  NONE  -Me caes muy bien \n-Tienes que jugar más part...
1  768213567418036224     N  @myendlesshazza a. que puto mal escribo\n\nb. ...
2  768212591105703936     N  @estherct209 jajajaja la tuya y la d mucha gen...
3  768221670255493120     P  Quiero mogollón a @AlbaBenito99 pero sobretodo...
4  768221021300264964     N  Vale he visto la tia bebiendose su regla y me ...


**Tokenizer**

In [103]:
from nltk.tokenize import TweetTokenizer

def tokenize(dataframe):
    dataframe['tokenized_text'] = dataframe['text'].map(
        TweetTokenizer(strip_handles=False, reduce_len=True, preserve_case=False).tokenize
    )

    dataframe['tokenized_text'] = dataframe['tokenized_text'].map(
        ' '.join
    )

    return dataframe

In [104]:
train_df = tokenize(train_df)
dev_df = tokenize(dev_df)
test_df = tokenize(test_df)

In [105]:
from sklearn.feature_extraction.text import CountVectorizer
import re

def vocab_reducer(text):
    res = []
    for word in text.split():
        word = re.sub('@.*','mention', word)
        word = re.sub('#(.*)', 'tag', word)
        word = re.sub('http.*', 'web', word)
        word = re.sub('\d.*', 'num', word)
        res.append(word)
    return (res)

vectorizer = CountVectorizer(tokenizer=vocab_reducer)

def vectorize(data, train=True, vectorizer=vectorizer):
    data_vec = vectorizer.fit_transform(data['text'])

    if not train:
        data_vec = vectorizer.transform(data['text'])

    return data_vec

In [106]:
train_vec = vectorize(train_df)
dev_vec = vectorize(dev_df, False)
test_vec = vectorize(test_df, False)

In [107]:
from nltk.stem import SnowballStemmer
spanish_stemmer = SnowballStemmer('spanish')

def stemmer(word):
    return spanish_stemmer.stem(word)

In [108]:
 
def load_polarity_lexicon(path):
    polarity_dict = {}
    with open(path,'r') as file:
        text = file.read()
        text = re.sub(r'#.*', '', text)
        for line in text.split('\n'):
            if len(line) >  0:
                word = re.split(r'(\t|\s+)',line)[0]
                tag = re.split(r'(\t|\s+)',line)[-1]
                polarity_dict[stemmer(word)] = tag
    
    return polarity_dict

In [109]:
polarity_dict = load_polarity_lexicon('./data/ElhPolar_esV1.lex')

In [110]:
print(polarity_dict.keys())

dict_keys(['a_cieg', 'a_flot', 'a_la_der', 'a_la_mod', 'a_la_sombr', 'a_pesar_d', 'a_salv', 'abandon', 'abarat', 'abat', 'abdic', 'aberr', 'abofet', 'abogar_por', 'abomin', 'abord', 'aborrec', 'abras', 'abraz', 'abrum', 'absolv', 'absorbent', 'absurd', 'abuch', 'abuche', 'abund', 'aburr', 'abusar_d', 'abus', 'abyect', 'acalor', 'acat', 'acced', 'acept', 'acert', 'achaqu', 'aciag', 'aclam', 'aclar', 'acobard', 'acogedor', 'acog', 'acoger_con_agr', 'acomod', 'acongoj', 'aconsej', 'acord', 'acort', 'acos', 'acritud', 'activ', 'acuchill', 'acuerd', 'acus', 'adapt', 'adecu', 'adherent', 'adher', 'adhesion', 'adiccion', 'adict', 'adivin', 'adjudic', 'admir', 'admit', 'adoctrin', 'ador', 'adorn', 'adul', 'adulter', 'adulterar_con_drog', 'adversari', 'advers', 'advertent', 'afabil', 'afabl', 'afan', 'afect', 'afectu', 'aficion', 'afin', 'afirm', 'afliccion', 'afligid', 'aflig', 'afortun', 'afrent', 'agasaj', 'agil', 'agit', 'agolp', 'agot', 'agotador', 'agrad', 'agradec', 'agrav', 'agravi', 'a

In [111]:
import numpy as np

def polarity_count(dataframe):
    res = np.zeros((dataframe['tokenized_text'].size, 2))
    for i, sentence in enumerate(dataframe['tokenized_text']):
        for word in sentence.split(' '):
            polarity = polarity_dict.get(stemmer(word), None)
            if polarity == None:
                continue
            if polarity == 'positive':
                res[i, 0] += 1
            if polarity == 'negative':
                res[i, 1] += 1
    return res

In [112]:
train_pol_mat = polarity_count(train_df)
dev_pol_mat = polarity_count(dev_df)
test_pol_mat = polarity_count(test_df)

In [113]:
print(train_pol_mat)
print(sum(train_pol_mat[:,0]))
print(sum(train_pol_mat[:,0]))

[[4. 0.]
 [1. 3.]
 [2. 3.]
 ...
 [0. 2.]
 [1. 1.]
 [0. 4.]]
1623.0
1623.0


In [114]:
import scipy

train_matrix = scipy.sparse.hstack((train_vec, train_pol_mat))
dev_matrix = scipy.sparse.hstack((dev_vec, dev_pol_mat))
test_matrix = scipy.sparse.hstack((test_vec, test_pol_mat))

In [130]:
from sklearn import svm
from sklearn.metrics import classification_report

svm_classif = svm.LinearSVC(C=0.1)
svm_classif.fit(train_matrix, train_df['label'])
preds = svm_classif.predict(dev_matrix)

print('accuracy: {}\n'.format(sum(preds==dev_df['label'])/len(dev_df['label'])))

print(classification_report(dev_df['label'], preds))

accuracy: 0.9861111111111112

              precision    recall  f1-score   support

           N       0.97      1.00      0.98       418
         NEU       0.99      0.96      0.98       133
        NONE       1.00      0.97      0.99       139
           P       1.00      0.99      0.99       318

    accuracy                           0.99      1008
   macro avg       0.99      0.98      0.98      1008
weighted avg       0.99      0.99      0.99      1008

